# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

## Reading from Bronze

In [0]:
df = spark.table("workspace.bronze.cust_info")

 
# Some Transformation


## Trim Spaces From strings

In [0]:
trim_exprs = [
    F.trim(F.col(c)).alias(c) if t == 'string' else F.col(c)
    for c, t in df.dtypes
]
df = df.select(*trim_exprs)

## Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

## Remove Records with Missing Customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

## Renaming Columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
     

## checks of dataframe

In [0]:
df.head(10)

# Writing Into Silver



In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

## Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10
     